## Cargar los datos ##

In [17]:
import numpy as np
import pandas as pd

In [18]:
df = pd.read_csv('data/features_30_sec.csv', skipinitialspace=True)

# eliminar filas que no tienen género
df = df.dropna(subset=['label']) 

#feature que queremos predecir
y = df['label'] 

#features para el entrenamiento
X = df.drop(columns=["label"]).copy()

# Convertir columnas binarias de texto en 0 y 1
text_columns = X.select_dtypes(include=['str', 'object', 'string']).columns

for col in text_columns:
    values = X[col].dropna().unique()

    if len(values) == 2:
        X[col] = X[col].map({
            values[0]: 0,
            values[1]: 1})
            
#asegurar que solo hay valores numéricos
X = X.select_dtypes(include=np.number)

print("Número de canciones:", len(df))
print("Número de features:", X.shape[1])
print("\nCanciones por género:")
print(y.value_counts())

df.head()


Número de canciones: 1000
Número de features: 58

Canciones por género:
label
blues        100
classical    100
country      100
disco        100
hiphop       100
jazz         100
metal        100
pop          100
reggae       100
rock         100
Name: count, dtype: int64


,filename,length,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,...,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,label
0,blues.00000.wav,661794,0.350088,0.088757,0.130228,0.002827,1784.165850,129774.064525,2002.449060,85882.761315,...,52.420910,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
1,blues.00001.wav,661794,0.340914,0.094980,0.095948,0.002373,1530.176679,375850.073649,2039.036516,213843.755497,...,55.356403,-0.731125,60.314529,0.295073,48.120598,-0.283518,51.106190,0.531217,45.786282,blues
2,blues.00002.wav,661794,0.363637,0.085275,0.175570,0.002746,1552.811865,156467.643368,1747.702312,76254.192257,...,40.598766,-7.729093,47.639427,-1.816407,52.382141,-3.439720,46.639660,-2.231258,30.573025,blues
3,blues.00003.wav,661794,0.404785,0.093999,0.141093,0.006346,1070.106615,184355.942417,1596.412872,166441.494769,...,44.427753,-3.319597,50.206673,0.636965,37.319130,-0.619121,37.259739,-3.407448,31.949339,blues
4,blues.00004.wav,661794,0.308526,0.087841,0.091529,0.002303,1835.004266,343399.939274,1748.172116,88445.209036,...,86.099236,-5.454034,75.269707,-0.916874,53.613918,-4.404827,62.910812,-11.703234,55.195160,blues


## Definición de los modelos ##

Un pipeline es una cadena de pasos que se ejecutan en orden. Por ejemplo, antes de entrenar un modelo, podemos necesitar rellenar datos que faltan y normalizar las características.

`Pipeline` permite crear un pipeline combinando varios pasos

`SimpleImputer` sirve para rellenar los valores que faltan en los datos, por ejemplo, cuando una característica musical contiene NaN. se puede usar `StandardScaler` para rellenar esos datos vacíos

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

#modelos
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

In [20]:
models = {
    "Logistic Regression": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000))
        ]),
        {
            "model__C": [0.01, 0.03, 0.1, 0.3, 1]
        }
    ),

    "k-NN": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier())
        ]),
        {
            "model__n_neighbors": [3, 5, 7, 9, 13, 17],
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2]
        }
    ),

    "Decision Tree": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DecisionTreeClassifier(random_state=42))
        ]),
        {
            "model__max_depth": [5, 10, 15],
            "model__min_samples_split": [5, 10],
            "model__min_samples_leaf": [5, 10]
        }
    ),

    "Artificial Neural Network": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                max_iter=500,
                random_state=42,
                early_stopping=False,
                n_iter_no_change=15,
                tol=1e-3
            ))
        ]),
        {
            "model__hidden_layer_sizes": [(30,), (50,), (100,), (50, 25)],
            "model__alpha": [0.01, 0.1, 1.0]
        }
    )
}

## Definir la validación cruzada
 
Mismo esquema de 10 folds para comparar todos los algoritmos

In [21]:
from sklearn.model_selection import StratifiedKFold

In [22]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42)

## Entrenar y evaluar los modelos

Esta es la celda principal. `GridSearchCV` probará todas las configuraciones usando validación cruzada de 10 folds y seleccionará la que obtenga la mejor accuracy media para cada algoritmo

In [23]:
from sklearn.model_selection import GridSearchCV

In [24]:

results = []
best_models = {}

for name, (pipeline, params) in models.items():

    print("\nEvaluando:", name)

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring="accuracy",
        cv=cv,
        return_train_score=True,
        n_jobs=-1)

    search.fit(X, y)

    best_models[name] = search

    # Guardar los resultados de todas las configuraciones
    for i in range(len(search.cv_results_["params"])):
        results.append({
            "Algorithm": name,
            "Parameters": search.cv_results_["params"][i],
            "Train accuracy": search.cv_results_["mean_train_score"][i],
            "CV accuracy": search.cv_results_["mean_test_score"][i]})

    # Seleccionar la configuración con mayor accuracy
    # entre las que tienen overfitting < 0.15
    cv_results = pd.DataFrame(search.cv_results_)

    cv_results["Overfitting gap"] = (cv_results["mean_train_score"] - cv_results["mean_test_score"])

    valid_results = cv_results[cv_results["Overfitting gap"] < 0.15]

    #si no se alcanza el objetivo de ovefitting menor que 0,15, busca el que más ccuracy tenga directamente
    if not valid_results.empty:
        best_row = valid_results.sort_values("mean_test_score", ascending=False).iloc[0]
    else:
        best_row = cv_results.sort_values("mean_test_score", ascending=False).iloc[0]

    print("Best parameters:", best_row["params"])
    print("Best 10-fold CV accuracy (without overfitting):", round(best_row["mean_test_score"], 4))
    print("Overfitting gap:", round(best_row["Overfitting gap"], 4))
    print("Overfitting ok:", best_row["Overfitting gap"] < 0.15)


results_df = pd.DataFrame(results)

results_df["Overfitting gap"] = (
    results_df["Train accuracy"] -
    results_df["CV accuracy"]
)

results_df["Acceptable accuracy"] = (
    (results_df["CV accuracy"] > 0.70) 
)

results_df["Acceptable overfitting"] = (
    results_df["Overfitting gap"] < 0.15
)
results_df["Fits objectives"] = (
    (results_df["Overfitting gap"] < 0.15) & 
    (results_df["CV accuracy"] > 0.70)
)

results_df = results_df.sort_values(
    ["Acceptable overfitting", "CV accuracy", "Overfitting gap"],
    ascending=[False, False, True]
).reset_index(drop=True)

print(f"Total configs evaluated: {len(results_df)}")
print(f"Configs with appropiate overfitting: "
      f"{results_df['Acceptable overfitting'].sum()}")
print(f"Configs with both good accuracy and overfitting: "
      f"{results_df['Fits objectives'].sum()}")

display(results_df)



Evaluando: Logistic Regression


Best parameters: {'model__C': 1}
Best 10-fold CV accuracy (without overfitting): 0.737
Overfitting gap: 0.1486
Overfitting ok: True

Evaluando: k-NN
Best parameters: {'model__n_neighbors': 3, 'model__p': 1, 'model__weights': 'uniform'}
Best 10-fold CV accuracy (without overfitting): 0.707
Overfitting gap: 0.1411
Overfitting ok: True

Evaluando: Decision Tree
Best parameters: {'model__max_depth': 5, 'model__min_samples_leaf': 5, 'model__min_samples_split': 5}
Best 10-fold CV accuracy (without overfitting): 0.555
Overfitting gap: 0.091
Overfitting ok: True

Evaluando: Artificial Neural Network
Best parameters: {'model__alpha': 1.0, 'model__hidden_layer_sizes': (100,)}
Best 10-fold CV accuracy (without overfitting): 0.757
Overfitting gap: 0.2226
Overfitting ok: False
Total configs evaluated: 53
Configs with appropiate overfitting: 20
Configs with both good accuracy and overfitting: 5


,Algorithm,Parameters,Train accuracy,CV accuracy,Overfitting gap,Acceptable accuracy,Acceptable overfitting,Fits objectives
0,Logistic Regression,{'model__C': 1},0.885556,0.737,0.148556,True,True,True
1,Logistic Regression,{'model__C': 0.3},0.859222,0.731,0.128222,True,True,True
2,Logistic Regression,{'model__C': 0.1},0.830444,0.730,0.100444,True,True,True
3,k-NN,"{'model__n_neighbors': 3, 'model__p': 1, 'mode...",0.848111,0.707,0.141111,True,True,True
4,Logistic Regression,{'model__C': 0.03},0.779111,0.704,0.075111,True,True,True
5,k-NN,"{'model__n_neighbors': 5, 'model__p': 1, 'mode...",0.818333,0.700,0.118333,False,True,False
6,k-NN,"{'model__n_neighbors': 7, 'model__p': 1, 'mode...",0.795111,0.693,0.102111,False,True,False
7,k-NN,"{'model__n_neighbors': 9, 'model__p': 1, 'mode...",0.775111,0.688,0.087111,False,True,False
8,k-NN,"{'model__n_neighbors': 7, 'model__p': 2, 'mode...",0.780556,0.687,0.093556,False,True,False
9,k-NN,"{'model__n_neighbors': 9, 'model__p': 2, 'mode...",0.761222,0.679,0.082222,False,True,False
